In [ ]:
!pip install pymupdf opencv-python pillow matplotlib pandas

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import fitz

pdf_path = list(uploaded.keys())[0]

doc = fitz.open(pdf_path)

page_output_dir = "outputs/pages"
os.makedirs(page_output_dir, exist_ok=True)

for page_index in range(len(doc)):
    page = doc[page_index]

    matrix = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=matrix)

    page_number = page_index + 1
    image_path = f"{page_output_dir}/page_{page_number:03}.png"

    pix.save(image_path)

    print(f"{page_number}/{len(doc)} 페이지 이미지 저장 완료")

print("전체 페이지 이미지 변환 완료")
print("전체 페이지 수:", len(doc))

In [ ]:
page_files = sorted([
    f for f in os.listdir(page_output_dir)
    if f.endswith(".png")
])

print("생성된 페이지 이미지 개수:", len(page_files))
print("처음 5개:", page_files[:5])
print("마지막 5개:", page_files[-5:])

In [ ]:
import cv2
import json
import numpy as np

def detect_blocks(image_path):
    image = cv2.imread(image_path)
    page_height, page_width = image.shape[:2]

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 글자와 선을 강조하기 위한 이진화
    _, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

    # 글자와 선을 블록 단위로 묶기
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 10))
    dilated = cv2.dilate(binary, kernel, iterations=2)

    contours, _ = cv2.findContours(
        dilated,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    blocks = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)

        # 너무 작은 잡음 제거
        if w > 50 and h > 30:
            blocks.append({
                "bbox": [int(x), int(y), int(x + w), int(y + h)],
                "width": int(w),
                "height": int(h)
            })

    # 위에서 아래, 왼쪽에서 오른쪽 순서로 정렬
    blocks = sorted(blocks, key=lambda b: (b["bbox"][1], b["bbox"][0]))

    return blocks, page_width, page_height

In [ ]:
def is_inside(inner_bbox, outer_bbox):
    ix1, iy1, ix2, iy2 = inner_bbox
    ox1, oy1, ox2, oy2 = outer_bbox

    return ix1 >= ox1 and iy1 >= oy1 and ix2 <= ox2 and iy2 <= oy2


def remove_duplicate_blocks(blocks):
    filtered_blocks = []

    for i, block in enumerate(blocks):
        bbox = block["bbox"]
        inside_other = False

        for j, other_block in enumerate(blocks):
            if i == j:
                continue

            other_bbox = other_block["bbox"]

            # 현재 박스가 다른 큰 박스 안에 거의 포함되면 제거
            if is_inside(bbox, other_bbox):
                current_area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
                other_area = (other_bbox[2] - other_bbox[0]) * (other_bbox[3] - other_bbox[1])

                if current_area < other_area * 0.8:
                    inside_other = True
                    break

        if not inside_other:
            filtered_blocks.append(block)

    return filtered_blocks

In [ ]:
def classify_block(block, page_width, page_height):
    x1, y1, x2, y2 = block["bbox"]
    w = x2 - x1
    h = y2 - y1

    width_ratio = w / page_width
    height_ratio = h / page_height
    y_ratio = y1 / page_height

    # 1. 하단 페이지 번호/단원명
    if y_ratio > 0.9:
        return "footer"

    # 2. 표/그래프/큰 이미지 영역
    # 가로로 넓고 세로도 큰 경우
    if width_ratio > 0.55 and height_ratio > 0.18:
        return "table_or_figure"

    # 3. 표 제목/캡션 영역
    # 표 바로 위에 있는 짧은 줄 형태
    if width_ratio < 0.35 and height_ratio < 0.05 and y_ratio > 0.35:
        return "caption_or_label"

    # 4. 수식 또는 예시 박스 영역
    # 가운데 정렬되어 있고, 높이가 너무 크지 않은 박스
    center_x = (x1 + x2) / 2
    page_center_x = page_width / 2

    if abs(center_x - page_center_x) < page_width * 0.25:
        if 0.25 < width_ratio < 0.65 and height_ratio < 0.15:
            return "formula_or_example_box"

    # 5. 상단의 진짜 제목/헤더 영역
    # 제목은 페이지 맨 위쪽에 있고, 높이가 작으며, 너무 넓은 본문 박스는 제외함
    if y_ratio < 0.15 and height_ratio < 0.09 and 0.45 < width_ratio < 0.75:
        return "title_or_header"

    # 6. 작은 장식 요소
    if width_ratio < 0.15 and height_ratio < 0.08:
        return "small_element"

    # 7. 나머지는 본문
    return "text"

In [ ]:
layout_output_dir = "outputs/layout_results"
os.makedirs(layout_output_dir, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(page_output_dir)
    if f.endswith(".png")
])

print("구조화 대상 페이지 수:", len(image_files))

for idx, file_name in enumerate(image_files):
    image_path = os.path.join(page_output_dir, file_name)

    page_number = int(file_name.replace("page_", "").replace(".png", ""))

    blocks, page_width, page_height = detect_blocks(image_path)

    # 중복 블록 제거
    blocks = remove_duplicate_blocks(blocks)

    structured_blocks = []

    for block_idx, block in enumerate(blocks):
        block_type = classify_block(block, page_width, page_height)

        structured_blocks.append({
            "block_id": f"p{page_number:03}_b{block_idx + 1:03}",
            "type": block_type,
            "bbox": block["bbox"]
        })

    page_result = {
        "page_id": page_number,
        "image_path": image_path,
        "page_width": page_width,
        "page_height": page_height,
        "block_count": len(structured_blocks),
        "blocks": structured_blocks
    }

    save_path = f"{layout_output_dir}/page_{page_number:03}_layout.json"

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(page_result, f, ensure_ascii=False, indent=2)

    print(f"{idx + 1}/{len(image_files)} 구조화 완료: {save_path}")

print("전체 페이지 구조화 완료")

In [ ]:
layout_files = sorted([
    f for f in os.listdir(layout_output_dir)
    if f.endswith(".json")
])

print("구조화 JSON 파일 개수:", len(layout_files))
print("처음 5개:", layout_files[:5])
print("마지막 5개:", layout_files[-5:])

In [ ]:
import pandas as pd

summary_data = []

for file_name in layout_files:
    json_path = os.path.join(layout_output_dir, file_name)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    type_counts = {}

    for block in data["blocks"]:
        block_type = block["type"]
        type_counts[block_type] = type_counts.get(block_type, 0) + 1

    summary_data.append({
        "page_id": data["page_id"],
        "block_count": data["block_count"],
        "title_or_header": type_counts.get("title_or_header", 0),
        "text": type_counts.get("text", 0),
        "table_or_figure": type_counts.get("table_or_figure", 0),
        "footer": type_counts.get("footer", 0),
        "small_element": type_counts.get("small_element", 0)
    })

summary_df = pd.DataFrame(summary_data)

summary_path = "outputs/layout_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("구조화 요약 CSV 저장 완료:", summary_path)
summary_df.head()

In [ ]:
sample_json_path = "outputs/layout_results/page_008_layout.json"

with open(sample_json_path, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print(json.dumps(sample_data, ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt

sample_page = 8
sample_image_path = f"outputs/pages/page_{sample_page:03}.png"
sample_json_path = f"outputs/layout_results/page_{sample_page:03}_layout.json"

image = cv2.imread(sample_image_path)

with open(sample_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

for block in data["blocks"]:
    x1, y1, x2, y2 = block["bbox"]
    block_id = block["block_id"]

    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)
    cv2.putText(
        image,
        block_id,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

layout_sample_dir = "outputs/layout_sample"
os.makedirs(layout_sample_dir, exist_ok=True)

save_image_path = f"{layout_sample_dir}/page_{sample_page:03}_layout_visualization.png"
cv2.imwrite(save_image_path, image)

print("시각화 이미지 저장 완료:", save_image_path)

plt.figure(figsize=(10, 14))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import shutil

layout_sample_dir = "outputs/layout_sample"
os.makedirs(layout_sample_dir, exist_ok=True)

sample_pages = [8, 16]

for page_number in sample_pages:
    src = f"outputs/layout_results/page_{page_number:03}_layout.json"
    dst = f"{layout_sample_dir}/page_{page_number:03}_layout.json"

    shutil.copy(src, dst)

    print("샘플 JSON 복사 완료:", dst)

## 5단계 결과 정리

본 단계에서는 4단계에서 구현한 문서 구조화 방식을 전체 PDF 페이지에 적용하였다.

PyMuPDF를 이용하여 원본 PDF 전체 198페이지를 페이지 단위 이미지로 변환한 뒤, OpenCV 기반 블록 감지 알고리즘을 전체 페이지 이미지에 적용하였다.

각 페이지에서는 흑백 변환, 이진화, dilation, contour detection 과정을 통해 문서 요소 후보 영역을 탐지하였다. 탐지된 블록에는 page_id와 block_id를 부여하였고, bbox 좌표를 [x1, y1, x2, y2] 형식으로 저장하였다.

또한 블록의 위치, 너비 비율, 높이 비율, 중앙 정렬 여부를 기준으로 title_or_header, text, formula_or_example_box, caption_or_label, table_or_figure, footer, small_element 등의 임시 유형을 부여하였다.

전체 198페이지에 대해 구조화 JSON 파일을 생성하였으며, 각 페이지별 블록 개수와 유형별 개수를 정리한 layout_summary.csv 파일을 생성하였다.

초기 구조화 결과에서는 일부 본문 영역이 title_or_header로 분류되는 오분류가 확인되었다. 이를 보완하기 위해 블록의 위치뿐 아니라 너비 비율, 높이 비율, 중앙 정렬 여부를 함께 고려하도록 분류 기준을 수정하였다.

보완 후 8페이지 구조화 결과에서 본문 영역은 text로, 예시 박스는 formula_or_example_box로, 표 제목은 caption_or_label로, 표 영역은 table_or_figure로 분리되어 보다 자연스러운 구조화 결과를 확인하였다.

이후 단계에서는 생성된 구조화 결과를 바탕으로 중복 블록, 누락 영역, 표/본문 오분류 여부를 평가하는 evaluation을 수행할 예정이다.